# Résumé détaillé — Notebook DataDowload&Preparation : Setup, Données, Prétraitement (Modules 0 à 3)

## Contexte général

Ce notebook construit la fondation du pipeline d'analyse multi-omique rigoureux (TCGA-BRCA : mutations, CNV, mRNA, RPPA → PAM50). Son objectif est de préparer les données et de construire un cache de représentations réduites, réutilisable par tous les modules d'analyse suivants (Component I à V), sans jamais avoir à refaire ce travail coûteux.

---

## MODULE 0 — Setup, configuration, provenance

**Ce qu'il fait :**
- Importe toutes les bibliothèques nécessaires (numpy, pandas, scikit-learn, xgboost, scipy, statsmodels).
- Monte Google Drive et définit `DRIVE_DIR` (`/content/drive/MyDrive/fati/TCGA_BRCA_data`).
- Crée un dossier de run daté (`RUN_ID`, ex. `2026-07-26`) sous `DRIVE_DIR/runs/`, avec ses sous-dossiers `checkpoints/` et `results/`.
- Définit `CONFIG` : tous les paramètres verrouillés du pipeline (nombre de folds CV, nombre de répétitions, seuils de filtrage, mode de réduction dimensionnelle, etc.).
- Fige une copie immuable de cette configuration (`CONFIG_CANONICAL`), pour que les sauvegardes/rechargements de checkpoints restent cohérents même si `CONFIG` est temporairement modifié plus tard (sweep de sensibilité).
- Définit les fonctions utilitaires : `log()` (journal + affichage), `ckpt_save()`/`ckpt_load()` (sérialisation pickle avec vérification de compatibilité de configuration), `NpEncoder` (pour sérialiser correctement les types numpy en JSON plus tard).
- Met en place un **marqueur de run incomplet** (`_INCOMPLETE`) sur Drive, pour éviter de démarrer accidentellement un nouveau run vide si un run précédent n'a pas été terminé.

**Pourquoi :** cette étape garantit la traçabilité (quel run, quelle config, quel environnement logiciel) et la reprise sécurisée en cas d'interruption — fondations indispensables avant tout calcul.

---

## MODULE 1 — Chargement et alignement des données brutes

**Ce qu'il fait, dans l'ordre :**

1. **mRNA** : chargé depuis `mrna_hugo_mapped.parquet` (le fichier déjà préparé manuellement en amont, contenant le mapping Ensembl→Hugo Symbol), plutôt que depuis le `.gz` brut. C'est l'override qu'on a ajouté spécifiquement.

2. **CNV** : chargé depuis `Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz`, transposé (patients en lignes), identifiants patients normalisés (`TCGA-XX-XXXX`, suffixes `-01` retirés).

3. **RPPA** : chargé depuis `RPPA_RBN.gz`, même traitement.

4. **Mutations + PAM50** : extraits directement de l'archive `brca_tcga_pan_can_atlas_2018.tar.gz` (sans décompression préalable sur disque) :
   - `data_mutations.txt` : filtré aux variantes **non-synonymes** uniquement (Missense, Nonsense, Frame_Shift, etc.), puis transformé en matrice binaire gène × patient (présence/absence de mutation).
   - `data_clinical_patient.txt` : extraction de la colonne `SUBTYPE`, nettoyage du préfixe `BRCA_`, filtrage aux 5 sous-types PAM50 valides (LumA, LumB, Her2, Basal, Normal).

5. **Intersection stricte des patients** : seuls les patients présents dans **les 4 couches simultanément ET** ayant un label PAM50 valide sont conservés.

6. **Encodage des labels** (`LabelEncoder`) et vérification finale : au moins autant de patients par classe que de folds CV prévus.

**Résultat obtenu (confirmé par exécution) :**
```
681 patients, 4 couches complètes
Classes : Basal=119, Her2=64, LumA=323, LumB=156, Normal=19
Mutations : (681, 16432)
CNV       : (681, 24776)
mRNA      : (681, 44292)
RPPA      : (681, 131)
```

**Pourquoi c'est important :** cette répartition de classes correspond exactement à celle du manuscrit original (LumA 47.4%, LumB 22.9%, Basal 17.5%, Her2 9.4%, Normal 2.8%) — confirmation forte que la reconstruction des données est fidèle à l'analyse originale.

---

## MODULE 2 — Découpes de validation croisée partagées

**Ce qu'il fait :**
- Génère **50 découpes** (`SPLITS`) via `RepeatedStratifiedKFold` : 10 répétitions × 5 folds, avec stratification (proportions de classes respectées dans chaque fold).
- Associe à chaque découpe un indice de répétition (`REPEAT_OF`), qui servira plus tard de graine aléatoire pour les classifieurs — permettant aux répétitions de balayer à la fois la variance de partition et la variance de graine.

**Pourquoi :** c'est le socle de la rigueur statistique Q1 — un seul découpage ne suffirait pas à estimer une variance fiable ; ces mêmes 50 découpes seront réutilisées **identiquement** par tous les modules suivants (tuning, évaluation, tests statistiques), garantissant que toutes les comparaisons entre classifieurs et configurations sont faites sur un pied d'égalité strict.

---

## MODULE 3 — Fonction de prétraitement unique + cache de réduction

**La fonction `preprocess_layer()` applique, dans l'ordre, et exclusivement sur les données d'entraînement (jamais sur le test) :**

1. **Imputation** : kNN (k=5) pour RPPA après filtrage des protéines à >20% de valeurs manquantes ; médiane pour les autres couches si nécessaire.
2. **Filtre de fréquence** (mutations uniquement) : conserve les gènes mutés entre 1% et 99% des patients (élimine les événements ubiquitaires ou trop rares).
3. **Filtre de variance quasi-nulle** : élimine les gènes non informatifs.
4. **Sélection top-5000 par dispersion** (CNV, mRNA) : utilise l'IQR (25% de point de rupture) plutôt que le MAD classique (50%), pour ne pas exclure des gènes variant uniquement dans les sous-types minoritaires.
5. **Standardisation** (z-score), sauf pour les mutations (binaires).
6. **Réduction dimensionnelle** : PCA (ou TruncatedSVD pour les mutations, binaires), en mode `fixed_variance` — le nombre de composantes est ajusté pour atteindre 80% de variance expliquée, plafonné à 120 composantes maximum.

**Ce cache est construit pour les 50 découpes × 4 couches = 200 combinaisons**, chacune sauvegardée de façon incrémentale (reprenable en cas d'interruption).

**Résultat obtenu (confirmé par exécution, 280 secondes) :**

| Couche | Features retenus | Composantes | Variance atteinte |
|---|---|---|---|
| Mutations | 1599 | 120 (plafonné) | 68.4% |
| CNV | 5000 | 16 | 80.4% |
| mRNA | 5000 | 120 (plafonné) | 79.3% |
| RPPA | 131 | 35 | 80.3% |

**Interprétation** : CNV et RPPA atteignent bien la cible de 80% avec peu de composantes (données à structure compressible). Mutations et mRNA plafonnent à 120 composantes sans tout à fait atteindre 80% — comportement attendu et documenté dès la conception du pipeline (couches à dimensionnalité intrinsèque plus élevée ou données éparses), pas un dysfonctionnement.

---

## Un bug rencontré et corrigé en cours de route

Lors de la première tentative d'exécution du module 3, une erreur `ValueError: too many values to unpack` s'est produite : la ligne d'assignation tentait de répartir 5 valeurs (`Xtr, Xte, nf, vr, nc`) sur seulement 4 cibles. Correction : regrouper `Xtr` et `Xte` dans un tuple `(Xtr, Xte)` assigné à `RED[(s, k)]`, puis assigner séparément `vr`, `nf`, `nc`. Une fois corrigé, le module a tourné sans erreur.

---

## État final à la fin du Notebook 1

- **Données** : 681 patients, 4 couches alignées, en mémoire (`RAW`, `y`, `le`, etc.) — mais ces variables **ne persistent pas** au-delà de la session.
- **Découpes CV** : 50 splits générés, déterministes (seed=42) — reproductibles à l'identique dans un nouveau notebook.
- **Cache de réduction** : sauvegardé sur Drive dans `ckpt_cache.pkl` sous `RUN_ID` — **ce résultat, lui, persiste** et pourra être rechargé sans recalcul.

## Transition vers le Notebook 2

Puisque les variables Python ne persistent pas entre sessions, le Notebook 2 devra **ré-exécuter rapidement les modules 0, 1, 2** (setup, chargement des données, génération des mêmes 50 splits — quelques minutes, pas de calcul lourd) pour reconstruire l'état en mémoire, puis **recharger le cache du module 3** directement depuis le checkpoint Drive (`ckpt_load("cache")`) au lieu de le recalculer, avant d'enchaîner sur le module 4 (tuning des hyperparamètres) et la suite du pipeline.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/fati/TCGA_BRCA_data/TCGA-BRCA.star_tpm.tsv"

# Lecture rapide des 5 premières lignes seulement, pour vérifier la structure
preview = pd.read_csv(path, sep="\t", index_col=0, nrows=5)
print("Shape (aperçu) :", preview.shape)
print("\nColonnes (patients), premières :")
print(preview.columns[:5].tolist())
print("\nIndex (gènes), premiers :")
print(preview.index[:5].tolist())
print("\nValeurs (aperçu) :")
print(preview.iloc[:5, :5])
print(f"\nMin/Max sur cet aperçu : {preview.values.min():.3f} / {preview.values.max():.3f}")

Shape (aperçu) : (5, 1226)

Colonnes (patients), premières :
['TCGA-D8-A146-01A', 'TCGA-AQ-A0Y5-01A', 'TCGA-C8-A274-01A', 'TCGA-BH-A0BD-01A', 'TCGA-B6-A1KC-01B']

Index (gènes), premiers :
['ENSG00000000003.15', 'ENSG00000000005.6', 'ENSG00000000419.13', 'ENSG00000000457.14', 'ENSG00000000460.17']

Valeurs (aperçu) :
                    TCGA-D8-A146-01A  TCGA-AQ-A0Y5-01A  TCGA-C8-A274-01A  \
Ensembl_ID                                                                 
ENSG00000000003.15          5.662037          3.703721          6.514515   
ENSG00000000005.6           3.376096          0.463099          0.000000   
ENSG00000000419.13          6.860140          7.086452          6.805072   
ENSG00000000457.14          4.400552          4.051007          5.037264   
ENSG00000000460.17          2.845169          2.426989          4.043248   

                    TCGA-BH-A0BD-01A  TCGA-B6-A1KC-01B  
Ensembl_ID                                              
ENSG00000000003.15          4.7849

In [ ]:
import pandas as pd

def strip_ensembl_version(gene_id):
    """ENSG00000000003.15 -> ENSG00000000003"""
    return gene_id.split(".")[0]

def map_ensembl_to_hugo(mrna_df, mapping_file=None):
    """
    Convertit les colonnes Ensembl Gene ID (versionnées) en symboles Hugo.
    mapping_file : table de correspondance (2 colonnes: ensembl_id, hugo_symbol)
                   Si absent, essaie de télécharger depuis mygene ou biomart.
    """
    # Étape 1 : retirer les versions
    mrna_df = mrna_df.copy()
    mrna_df.index = mrna_df.index.map(strip_ensembl_version)

    if mapping_file is not None:
        mapping = pd.read_csv(mapping_file, sep="\t")
        id_to_symbol = dict(zip(mapping["ensembl_id"], mapping["hugo_symbol"]))
    else:
        # Solution rapide : mygene.info API (nécessite internet, gère les batchs)
        import mygene
        mg = mygene.MyGeneInfo()
        unique_ids = mrna_df.index.unique().tolist()
        print(f"Requête mygene pour {len(unique_ids)} gènes...")
        results = mg.querymany(unique_ids, scopes="ensembl.gene",
                               fields="symbol", species="human")
        id_to_symbol = {r["query"]: r.get("symbol") for r in results if "symbol" in r}

    mrna_df["hugo_symbol"] = mrna_df.index.map(id_to_symbol)
    n_before = mrna_df.shape[0]
    mrna_df = mrna_df.dropna(subset=["hugo_symbol"])
    n_after = mrna_df.shape[0]
    print(f"Mapping : {n_after}/{n_before} gènes convertis en symboles Hugo "
          f"({n_before - n_after} perdus, sans correspondance)")

    mrna_df = mrna_df.set_index("hugo_symbol")
    mrna_df = mrna_df.groupby(mrna_df.index).mean()  # fusionne les doublons
    return mrna_df

In [ ]:
!pip install mygene --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 1.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from pathlib import Path

# Chemin exact de votre fichier déjà décompressé
path = "/content/drive/MyDrive/fati/TCGA_BRCA_data/TCGA-BRCA.star_tpm.tsv"

def patient_id(x):
    p = str(x).split("-")
    return "-".join(p[:3]) if len(p) >= 3 else str(x)

def read_matrix(path, transpose=True):
    df = pd.read_csv(path, sep="\t", index_col=0)
    if transpose:
        df = df.T
    df.index = df.index.map(patient_id)
    return df.groupby(df.index).mean()

print("Lecture du fichier complet (peut prendre 1-2 min, ~900 MB)...")
mrna = read_matrix(path)
print(f"Shape : {mrna.shape}  (patients x gènes)")
print(mrna.iloc[:5, :5])

Lecture du fichier complet (peut prendre 1-2 min, ~900 MB)...
Shape : (1095, 60660)  (patients x gènes)
Ensembl_ID    ENSG00000000003.15  ENSG00000000005.6  ENSG00000000419.13  \
TCGA-3C-AAAU            3.407747           0.125387            6.712645   
TCGA-3C-AALI            3.228680           0.167486            7.107211   
TCGA-3C-AALJ            5.348923           2.554785            7.208299   
TCGA-3C-AALK            5.446713           0.234379            6.243363   
TCGA-4H-AAAK            5.596917           0.887135            6.653371   

Ensembl_ID    ENSG00000000457.14  ENSG00000000460.17  
TCGA-3C-AAAU            3.358185            2.264056  
TCGA-3C-AALI            5.052007            3.080419  
TCGA-3C-AALJ            3.111148            2.709401  
TCGA-3C-AALK            3.748279            2.588900  
TCGA-4H-AAAK            3.727060            2.614686  


In [ ]:
!pip install mygene --quiet

import mygene
import numpy as np
import time

def strip_ensembl_version(gene_id):
    return gene_id.split(".")[0]

# Étape 1 : nettoyer les versions
gene_cols = mrna.columns.tolist()
stripped = [strip_ensembl_version(g) for g in gene_cols]
unique_ids = sorted(set(stripped))
print(f"{len(unique_ids)} identifiants Ensembl uniques à mapper")

# Étape 2 : requêtes par batch pour éviter timeout/surcharge
mg = mygene.MyGeneInfo()
BATCH_SIZE = 2000
id_to_symbol = {}

t0 = time.time()
for i in range(0, len(unique_ids), BATCH_SIZE):
    batch = unique_ids[i:i + BATCH_SIZE]
    try:
        results = mg.querymany(batch, scopes="ensembl.gene",
                               fields="symbol", species="human",
                               verbose=False)
        for r in results:
            if "symbol" in r:
                id_to_symbol[r["query"]] = r["symbol"]
    except Exception as e:
        print(f"  Batch {i}-{i+BATCH_SIZE} échoué : {e}")
    print(f"  Batch {i//BATCH_SIZE + 1}/{(len(unique_ids)-1)//BATCH_SIZE + 1} "
          f"terminé ({time.time()-t0:.0f}s écoulées, "
          f"{len(id_to_symbol)} mappés jusqu'ici)")

print(f"\nTotal mappé : {len(id_to_symbol)}/{len(unique_ids)} "
      f"({time.time()-t0:.0f}s)")

60616 identifiants Ensembl uniques à mapper
  Batch 1/31 terminé (5s écoulées, 1998 mappés jusqu'ici)
  Batch 2/31 terminé (9s écoulées, 3992 mappés jusqu'ici)
  Batch 3/31 terminé (13s écoulées, 5989 mappés jusqu'ici)
  Batch 4/31 terminé (17s écoulées, 7983 mappés jusqu'ici)
  Batch 5/31 terminé (21s écoulées, 9975 mappés jusqu'ici)
  Batch 6/31 terminé (25s écoulées, 11971 mappés jusqu'ici)
  Batch 7/31 terminé (30s écoulées, 13958 mappés jusqu'ici)
  Batch 8/31 terminé (34s écoulées, 15920 mappés jusqu'ici)
  Batch 9/31 terminé (38s écoulées, 17888 mappés jusqu'ici)
  Batch 10/31 terminé (43s écoulées, 19837 mappés jusqu'ici)
  Batch 11/31 terminé (47s écoulées, 21803 mappés jusqu'ici)
  Batch 12/31 terminé (51s écoulées, 23590 mappés jusqu'ici)
  Batch 13/31 terminé (55s écoulées, 25200 mappés jusqu'ici)
  Batch 14/31 terminé (59s écoulées, 26654 mappés jusqu'ici)
  Batch 15/31 terminé (63s écoulées, 28069 mappés jusqu'ici)
  Batch 16/31 terminé (67s écoulées, 29541 mappés jusqu'i

In [ ]:
# 1. Combien de gènes non mappés, et à quoi ressemblent-ils ?
mapped_ids = set(id_to_symbol.keys())
unmapped_ids = set(unique_ids) - mapped_ids
print(f"Non mappés : {len(unmapped_ids)}")
print(f"Exemples : {list(unmapped_ids)[:20]}")

# 2. Vérifier spécifiquement les gènes PAM50 -- CRITIQUE
PAM50_GENES = [
    "ACTR3B","ANLN","BAG1","BCL2","BIRC5","BLVRA","CCNB1","CCNE1","CDC20",
    "CDC6","CDH3","CENPF","CEP55","CXXC5","EGFR","ERBB2","ESR1","EXO1",
    "FGFR4","FOXA1","FOXC1","GPR160","GRB7","KIF2C","KRT14","KRT17","KRT5",
    "MAPT","MDM2","MELK","MIA","MKI67","MLPH","MMP11","MYBL2","MYC",
    "NAT1","NDC80","NUF2","ORC6","PGR","PHGDH","PTTG1","RRM2","SFRP1",
    "SLC39A6","TMEM45B","TYMS","UBE2C","UBE2T"
]
mapped_symbols = set(id_to_symbol.values())
pam50_present = [g for g in PAM50_GENES if g in mapped_symbols]
pam50_missing = [g for g in PAM50_GENES if g not in mapped_symbols]
print(f"\nGènes PAM50 mappés : {len(pam50_present)}/{len(PAM50_GENES)}")
if pam50_missing:
    print(f"PAM50 MANQUANTS : {pam50_missing}")

Non mappés : 15277
Exemples : ['ENSG00000226744', 'ENSG00000230990', 'ENSG00000263126', 'ENSG00000286713', 'ENSG00000256708', 'ENSG00000260618', 'ENSG00000251510', 'ENSG00000254424', 'ENSG00000271025', 'ENSG00000238165', 'ENSG00000279135', 'ENSG00000272582', 'ENSG00000251330', 'ENSG00000276505', 'ENSG00000226636', 'ENSG00000236507', 'ENSG00000259198', 'ENSG00000286711', 'ENSG00000285778', 'ENSG00000286738']

Gènes PAM50 mappés : 50/50


In [ ]:
new_cols = [id_to_symbol.get(g, None) for g in stripped]
n_before = mrna.shape[1]
keep_mask = [c is not None for c in new_cols]

mrna_mapped = mrna.loc[:, keep_mask].copy()
mrna_mapped.columns = [c for c in new_cols if c is not None]

n_after = mrna_mapped.shape[1]
print(f"Colonnes : {n_before} -> {n_after} après mapping")

# Fusionner les doublons de symboles Hugo (plusieurs Ensembl ID -> même symbole)
n_unique_symbols = len(set(mrna_mapped.columns))
print(f"Symboles uniques : {n_unique_symbols} "
      f"(doublons fusionnés : {n_after - n_unique_symbols})")
mrna_mapped = mrna_mapped.T.groupby(level=0).mean().T
print(f"Shape finale : {mrna_mapped.shape}")

# Vérification finale des PAM50 dans la matrice finale
present_final = [g for g in PAM50_GENES if g in mrna_mapped.columns]
print(f"\nPAM50 présents dans la matrice finale : {len(present_final)}/50")

Colonnes : 60660 -> 45377 après mapping
Symboles uniques : 44292 (doublons fusionnés : 1085)
Shape finale : (1095, 44292)

PAM50 présents dans la matrice finale : 50/50


In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")

# 1. Le mapping lui-même (réutilisable pour CNV si besoin, ou pour d'autres runs)
mapping_path = DATA_DIR / "ensembl_to_hugo_mapping.json"
with open(mapping_path, "w") as f:
    json.dump(id_to_symbol, f)
print(f"Mapping sauvegardé -> {mapping_path} ({len(id_to_symbol)} entrées)")

# 2. La matrice mRNA finale, prête à être utilisée dans le pipeline
mrna_out = DATA_DIR / "mrna_hugo_mapped.parquet"
mrna_mapped.to_parquet(mrna_out)
print(f"Matrice mRNA sauvegardée -> {mrna_out} ({mrna_mapped.shape})")

Mapping sauvegardé -> /content/drive/MyDrive/fati/TCGA_BRCA_data/ensembl_to_hugo_mapping.json (45339 entrées)
Matrice mRNA sauvegardée -> /content/drive/MyDrive/fati/TCGA_BRCA_data/mrna_hugo_mapped.parquet ((1095, 44292))


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")

FILES_TO_CHECK = {
    "CNV (GISTIC2)":       "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz",
    "RPPA":                "RPPA_RBN.gz",
    "Mutations + PAM50":   "brca_tcga_pan_can_atlas_2018.tar.gz",
}

print(f"Dossier vérifié : {DATA_DIR}\n")
print(f"{'Fichier':<25}{'Présent (.gz)':<16}{'Taille':<12}{'Décompressé':<14}")
print("-" * 67)

for label, fname in FILES_TO_CHECK.items():
    p_gz = DATA_DIR / fname
    exists_gz = p_gz.exists()
    size_mb = f"{p_gz.stat().st_size / 1e6:.1f} MB" if exists_gz else "—"

    # Pour les .tar.gz, on ne "décompresse" pas un seul fichier -- on vérifie
    # juste la présence de l'archive elle-même ici (extraction faite à la volée
    # par tarfile dans le pipeline, comme pour les mutations).
    if fname.endswith(".tar.gz"):
        decompressed_status = "N/A (extrait à la volée)"
    else:
        p_out = p_gz.with_suffix("")  # retire le .gz
        decompressed_status = "Oui" if p_out.exists() else "Non"

    print(f"{label:<25}{str(exists_gz):<16}{size_mb:<12}{decompressed_status:<14}")

print("\nListe complète du dossier :")
for f in sorted(DATA_DIR.glob("*")):
    print(f"  {f.name:<55} {f.stat().st_size/1e6:>10.1f} MB")

Dossier vérifié : /content/drive/MyDrive/fati/TCGA_BRCA_data

Fichier                  Présent (.gz)   Taille      Décompressé   
-------------------------------------------------------------------
CNV (GISTIC2)            False           —           Non           
RPPA                     False           —           Non           
Mutations + PAM50        False           —           N/A (extrait à la volée)

Liste complète du dossier :
  TCGA-BRCA.star_tpm.tsv                                       895.6 MB
  TCGA-BRCA.star_tpm.tsv.gz                                    342.3 MB
  ensembl_to_hugo_mapping.json                                   1.4 MB
  mrna_hugo_mapped.parquet                                     298.6 MB


In [ ]:
import urllib.request
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")

URLS = {
    "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz":
        "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz",
    "RPPA_RBN.gz":
        "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/RPPA_RBN.gz",
}

for fname, url in URLS.items():
    dest = DATA_DIR / fname
    if dest.exists():
        print(f"Déjà présent : {fname}")
        continue
    print(f"Téléchargement de {fname} depuis {url} ...")
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"  -> {dest} ({dest.stat().st_size/1e6:.1f} MB)")
    except Exception as e:
        print(f"  ÉCHEC : {e}")

Téléchargement de Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz depuis https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz ...
  -> /content/drive/MyDrive/fati/TCGA_BRCA_data/Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz (2.5 MB)
Téléchargement de RPPA_RBN.gz depuis https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/RPPA_RBN.gz ...
  -> /content/drive/MyDrive/fati/TCGA_BRCA_data/RPPA_RBN.gz (0.7 MB)


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")

for label, fname in [("CNV", "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz"),
                     ("RPPA", "RPPA_RBN.gz")]:
    path = DATA_DIR / fname
    print(f"\n=== {label} ===")
    preview = pd.read_csv(path, sep="\t", index_col=0, compression="gzip", nrows=5)
    print(f"Shape (aperçu) : {preview.shape}")
    print(f"Colonnes (patients), premières : {preview.columns[:3].tolist()}")
    print(f"Index (gènes/protéines), premiers : {preview.index[:5].tolist()}")
    print(preview.iloc[:5, :3])
    print(f"Min/Max sur l'aperçu : {preview.values.min():.3f} / {preview.values.max():.3f}")


=== CNV ===
Shape (aperçu) : (5, 1080)
Colonnes (patients), premières : ['TCGA-3C-AAAU-01', 'TCGA-3C-AALI-01', 'TCGA-3C-AALJ-01']
Index (gènes/protéines), premiers : ['ACAP3', 'ACTRT2', 'AGRN', 'ANKRD65', 'ATAD3A']
             TCGA-3C-AAAU-01  TCGA-3C-AALI-01  TCGA-3C-AALJ-01
Gene Symbol                                                   
ACAP3                  0.069           -1.008            -0.33
ACTRT2                 0.069           -1.008            -0.33
AGRN                   0.069           -1.008            -0.33
ANKRD65                0.069           -1.008            -0.33
ATAD3A                 0.069           -1.008            -0.33
Min/Max sur l'aperçu : -1.293 / 1.646

=== RPPA ===
Shape (aperçu) : (5, 747)
Colonnes (patients), premières : ['TCGA-A1-A0SF-01', 'TCGA-A1-A0SH-01', 'TCGA-A1-A0SJ-01']
Index (gènes/protéines), premiers : ['1433EPSILON', '4EBP1', '4EBP1PS65', '4EBP1PT37T46', '53BP1']
                    TCGA-A1-A0SF-01  TCGA-A1-A0SH-01  TCGA-A1-A0SJ-01
Sampl

In [ ]:
import subprocess

# 1. Cloner le dépôt (sans les gros fichiers LFS pour l'instant)
result = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/cBioPortal/datahub.git",
     "/content/datahub_tmp"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout[-2000:])
print("STDERR:", result.stderr[-2000:])

STDOUT: 
STDERR:   51% (6989/13702)
Updating files:  52% (7126/13702)
Updating files:  53% (7263/13702)
Updating files:  54% (7400/13702)
Updating files:  55% (7537/13702)
Updating files:  56% (7674/13702)
Updating files:  57% (7811/13702)
Updating files:  58% (7948/13702)
Updating files:  59% (8085/13702)
Updating files:  60% (8222/13702)
Updating files:  61% (8359/13702)
Updating files:  62% (8496/13702)
Updating files:  63% (8633/13702)
Updating files:  64% (8770/13702)
Updating files:  65% (8907/13702)
Updating files:  66% (9044/13702)
Updating files:  67% (9181/13702)
Updating files:  68% (9318/13702)
Updating files:  69% (9455/13702)
Updating files:  70% (9592/13702)
Updating files:  71% (9729/13702)
Updating files:  72% (9866/13702)
Updating files:  73% (10003/13702)
Updating files:  73% (10112/13702)
Updating files:  74% (10140/13702)
Updating files:  75% (10277/13702)
Updating files:  76% (10414/13702)
Updating files:  77% (10551/13702)
Updating files:  78% (10688/13702)
Updat

In [ ]:
import subprocess

# Vérifier si git-lfs est disponible, sinon l'installer
r0 = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
print("git-lfs version:", r0.stdout, r0.stderr)

git-lfs version: git-lfs/3.7.1 (GitHub; linux amd64; go 1.26.0)
 


In [ ]:
from pathlib import Path

src = Path("/content/datahub_tmp/public/brca_tcga_pan_can_atlas_2018")
print(f"Dossier existe : {src.exists()}\n")

if src.exists():
    files = sorted(src.glob("*"))
    print(f"{len(files)} fichiers trouvés :\n")
    for f in files:
        size = f.stat().st_size
        print(f"  {f.name:<55} {size:>12,} octets")
else:
    print("Dossier introuvable -- vérifier le chemin du clone")

Dossier existe : True

58 fichiers trouvés :

  LICENSE                                                          446 octets
  README.md                                                      1,541 octets
  case_lists                                                     4,096 octets
  data_armlevel_cna.txt                                        323,254 octets
  data_clinical_patient.txt                                    341,851 octets
  data_clinical_sample.txt                                     204,137 octets
  data_clinical_supp_hypoxia.txt                                25,055 octets
  data_cna.txt                                              60,303,945 octets
  data_cna_hg19.seg                                         10,176,257 octets
  data_gene_panel_matrix.txt                                    30,349 octets
  data_genetic_ancestry.txt                                     62,835 octets
  data_log2_cna.txt                                        166,062,981 octets
  data_methylation

In [ ]:
import tarfile
from pathlib import Path

src = Path("/content/datahub_tmp/public/brca_tcga_pan_can_atlas_2018")
dest_archive = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data/brca_tcga_pan_can_atlas_2018.tar.gz")

print("Recréation de l'archive .tar.gz (peut prendre 1-2 min, ~900 Mo au total)...")
with tarfile.open(dest_archive, "w:gz") as tar:
    tar.add(src, arcname="brca_tcga_pan_can_atlas_2018")

print(f"-> {dest_archive} ({dest_archive.stat().st_size/1e6:.1f} MB)")

Recréation de l'archive .tar.gz (peut prendre 1-2 min, ~900 Mo au total)...
-> /content/drive/MyDrive/fati/TCGA_BRCA_data/brca_tcga_pan_can_atlas_2018.tar.gz (503.1 MB)


In [ ]:
import tarfile

with tarfile.open(dest_archive, "r:gz") as tar:
    names = tar.getnames()
    check_files = ["brca_tcga_pan_can_atlas_2018/data_mutations.txt",
                  "brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt"]
    for f in check_files:
        print(f"{f:<60} {'trouvé' if f in names else 'MANQUANT'}")

brca_tcga_pan_can_atlas_2018/data_mutations.txt              trouvé
brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt       trouvé


In [ ]:
"""
=============================================================================
MULTI-OMICS REDUNDANCY / COMPLEMENTARITY PIPELINE
TCGA-BRCA : somatic mutations, CNV, mRNA, RPPA  ->  PAM50
=============================================================================
"""

# %% ========================================================================
# 0. SETUP, CONFIG, PROVENANCE
# ===========================================================================
import os, sys, json, pickle, time, gc, warnings, hashlib, platform, tarfile, shutil
from datetime import datetime
from pathlib import Path
from itertools import combinations
from collections import Counter
from math import factorial

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     GridSearchCV)
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon, spearmanr
from xgboost import XGBClassifier
import sklearn, xgboost, scipy, statsmodels

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")   # <-- CORRIGÉ
except Exception:
    DRIVE_DIR = Path("./TCGA_BRCA_data")

RAW_DIR = DRIVE_DIR / "raw"
PROC_DIR = DRIVE_DIR                        # <-- les .gz sont directement ici, pas dans /raw
PROC_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = os.environ.get("RUN_ID") or pd.Timestamp.now().strftime("%Y-%m-%d")
RUN_DIR = DRIVE_DIR / "runs" / RUN_ID
CKPT_DIR, OUT_DIR = RUN_DIR / "checkpoints", RUN_DIR / "results"
for d in (CKPT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "cv_folds": 5,
    "cv_repeats": 10,
    "seed": 42,
    "tuning_inner_folds": 3,
    "filter_stat": "iqr",
    "filter_top_k": 5000,
    "min_variance": 1e-6,
    "mut_freq_lo": 0.01,
    "mut_freq_hi": 0.99,
    "rppa_max_missing": 0.20,
    "rppa_knn_k": 5,
    "dim_mode": "fixed_variance",
    "dim_variance_target": 0.80,
    "dim_k": 50,
    "dim_k_max": 120,
    "n_boot_rpred": 200,
    "n_boot_pid": 500,
    "boot_rf_trees": 200,
    "fdr_alpha": 0.05,
    "panel_delta": 0.03,
    "panel_rho_min": 0.70,
    "pid_bins": 10,
    "pid_n_pcs": 5,
    "pid_discretisation": "quantile",
    "metabric_top_genes": [100, 150, 200, 300, 500],
}
SEED = CONFIG["seed"]

import copy as _copy
CONFIG_CANONICAL = _copy.deepcopy(CONFIG)


class cfg_override:
    def __init__(self, **kw):
        self.kw, self.saved = kw, {}
    def __enter__(self):
        for k, v in self.kw.items():
            self.saved[k] = CONFIG[k]
            CONFIG[k] = v
        return CONFIG
    def __exit__(self, *exc):
        CONFIG.update(self.saved)
        return False

LAYERS = ["mutations", "cnv", "mrna", "rppa"]
SHORT = {"mutations": "Mutations", "cnv": "CNV", "mrna": "mRNA", "rppa": "RPPA"}
BINARY_LAYERS = {"mutations"}

RECIPE = {
    "mutations": (False, False, True, False),
    "cnv":       (True,  True,  False, False),
    "mrna":      (True,  True,  False, False),
    "rppa":      (True,  False, False, True),
}

NONSYNONYMOUS = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins", "Splice_Site",
    "Nonstop_Mutation", "Translation_Start_Site",
}

ENV = {"python": sys.version.split()[0], "platform": platform.platform(),
       "numpy": np.__version__, "pandas": pd.__version__,
       "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
       "scipy": scipy.__version__, "statsmodels": statsmodels.__version__}
INPUTS = {}
LOG = []
VAR_MISS = {}


class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return None if np.isnan(o) else float(o)
        if isinstance(o, (np.bool_,)):
            return bool(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (pd.Timestamp, datetime)):
            return o.isoformat(timespec="seconds")
        if isinstance(o, Path):
            return str(o)
        return super().default(o)


def log(msg):
    print(msg)
    LOG.append(f"{datetime.now().strftime('%H:%M:%S')}  {msg}")


def ckpt_save(name, data):
    with open(CKPT_DIR / f"ckpt_{name}.pkl", "wb") as f:
        pickle.dump({"data": data, "run": RUN_ID, "config": CONFIG_CANONICAL,
                     "env": ENV, "inputs": INPUTS,
                     "written": pd.Timestamp.now().isoformat(timespec="seconds")}, f)


def ckpt_load(name):
    p = CKPT_DIR / f"ckpt_{name}.pkl"
    if not p.exists():
        return None
    with open(p, "rb") as f:
        pl = pickle.load(f)
    if isinstance(pl, dict) and "run" in pl:
        if pl["config"] != CONFIG_CANONICAL:
            log(f"  [ckpt] {name} WAS WRITTEN UNDER A DIFFERENT CONFIG -- ignoring it")
            return None
        log(f"  [ckpt] {name} reloaded")
        return pl["data"]
    return pl


MARKER = DRIVE_DIR / "runs" / "_INCOMPLETE"
if "RUN_ID" not in os.environ and MARKER.exists():
    prev = MARKER.read_text().strip()
    if prev and prev != RUN_ID:
        log("!" * 72)
        log(f"An INCOMPLETE run exists: {prev}")
        log(f'    import os; os.environ["RUN_ID"] = "{prev}"')
        log("!" * 72)
MARKER.parent.mkdir(parents=True, exist_ok=True)
MARKER.write_text(RUN_ID)

log("=" * 72)
log(f"RUN_ID {RUN_ID}   ->  {RUN_DIR}")
log("=" * 72)


# %% ========================================================================
# 1. RAW DATA -- utilise les fichiers déjà téléchargés/décompressés
# ===========================================================================
def patient_id(x):
    p = str(x).split("-")
    return "-".join(p[:3]) if len(p) >= 3 else str(x)


def read_matrix_gz(path, transpose=True):
    df = pd.read_csv(path, sep="\t", index_col=0, compression="gzip")
    if transpose:
        df = df.T
    df.index = df.index.map(patient_id)
    return df.groupby(df.index).mean()


MRNA_MAPPED_OVERRIDE = DRIVE_DIR / "mrna_hugo_mapped.parquet"

RAW = {}

# ---- mRNA : utiliser le mapping Hugo déjà calculé ----
if MRNA_MAPPED_OVERRIDE.exists():
    log(f"  [override] mrna chargé depuis le mapping Hugo pré-calculé : {MRNA_MAPPED_OVERRIDE}")
    RAW["mrna"] = pd.read_parquet(MRNA_MAPPED_OVERRIDE)
else:
    RAW["mrna"] = read_matrix_gz(DRIVE_DIR / "TCGA-BRCA.star_tpm.tsv.gz")

# ---- CNV ----
RAW["cnv"] = read_matrix_gz(DRIVE_DIR / "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz")

# ---- RPPA ----
RAW["rppa"] = read_matrix_gz(DRIVE_DIR / "RPPA_RBN.gz")

# ---- Mutations + PAM50 (depuis l'archive tar.gz recréée) ----
tar_path = DRIVE_DIR / "brca_tcga_pan_can_atlas_2018.tar.gz"
with tarfile.open(tar_path, "r:gz") as tar:
    maf = pd.read_csv(tar.extractfile("brca_tcga_pan_can_atlas_2018/data_mutations.txt"),
                      sep="\t", comment="#", low_memory=False)
    maf = maf[maf["Variant_Classification"].isin(NONSYNONYMOUS)]
    maf["pid"] = maf["Tumor_Sample_Barcode"].map(patient_id)
    mut = (maf.groupby(["pid", "Hugo_Symbol"]).size()
              .unstack(fill_value=0).clip(upper=1).astype(np.int8))
    RAW["mutations"] = mut

    clin = pd.read_csv(tar.extractfile("brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt"),
                       sep="\t", comment="#", low_memory=False)

col = "SUBTYPE" if "SUBTYPE" in clin.columns else None
assert col, f"no SUBTYPE column; available: {list(clin.columns)[:25]}"
lab = clin.set_index("PATIENT_ID")[col].dropna().astype(str)
lab = lab.str.replace("^BRCA_", "", regex=True)
lab = lab[lab.isin(["LumA", "LumB", "Her2", "Basal", "Normal"])]
lab.index = lab.index.map(patient_id)
lab.name = "PAM50"
labels_all = lab

# ---- Intersection des patients communs à toutes les couches ----
common = set(labels_all.index)
for df in RAW.values():
    common &= set(df.index)
common = sorted(common)
RAW = {k: v.loc[common] for k, v in RAW.items()}
labels = labels_all.loc[common]

le = LabelEncoder()
y = le.fit_transform(labels.values)
N, K = len(y), len(le.classes_)

log(f"\nCohort: {N} patients with complete data across {len(LAYERS)} layers")
log(f"Classes: {dict(zip(le.classes_, np.bincount(y).tolist()))}")
for k in LAYERS:
    log(f"  {SHORT[k]:<12}{str(RAW[k].shape):>16}")
assert np.bincount(y).min() >= CONFIG["cv_folds"], \
    "a class has fewer members than the number of folds"

print("\n✅ Module 0-1 terminé : données chargées et alignées.")

RUN_ID 2026-07-26   ->  /content/drive/MyDrive/fati/TCGA_BRCA_data/runs/2026-07-26
  [override] mrna chargé depuis le mapping Hugo pré-calculé : /content/drive/MyDrive/fati/TCGA_BRCA_data/mrna_hugo_mapped.parquet

Cohort: 681 patients with complete data across 4 layers
Classes: {'Basal': 119, 'Her2': 64, 'LumA': 323, 'LumB': 156, 'Normal': 19}
  Mutations       (681, 16432)
  CNV             (681, 24776)
  mRNA            (681, 44292)
  RPPA              (681, 131)

✅ Module 0-1 terminé : données chargées et alignées.


In [ ]:
# %% ========================================================================
# 2. SHARED CV SPLITS
# ===========================================================================
NF, NR = CONFIG["cv_folds"], CONFIG["cv_repeats"]
SPLITS = list(RepeatedStratifiedKFold(n_splits=NF, n_repeats=NR,
                                      random_state=SEED).split(np.zeros(N), y))
REPEAT_OF = [i // NF for i in range(len(SPLITS))]
log(f"\n{len(SPLITS)} splits = {NR} repeats x {NF} folds "
    f"(classifier seed = repeat index)")


# %% ========================================================================
# 3. THE single PREPROCESSING FUNCTION
# ===========================================================================
def score_features(V_df, stat):
    if stat == "mad":
        med = V_df.median()
        return (V_df - med).abs().median()
    if stat == "iqr":
        q = V_df.quantile([0.25, 0.75])
        return q.loc[0.75] - q.loc[0.25]
    if stat == "var":
        return V_df.var()
    raise ValueError(stat)


def reduce_fit(Vtr, layer, seed):
    Reducer = TruncatedSVD if layer in BINARY_LAYERS else PCA
    if CONFIG["dim_mode"] == "fixed_k":
        k = min(CONFIG["dim_k"], Vtr.shape[1] - 1, Vtr.shape[0] - 1)
        return Reducer(n_components=k, random_state=seed).fit(Vtr), k
    kmax = min(CONFIG["dim_k_max"], Vtr.shape[1] - 1, Vtr.shape[0] - 1)
    rd = Reducer(n_components=kmax, random_state=seed).fit(Vtr)
    cum = np.cumsum(rd.explained_variance_ratio_)
    k = int(np.searchsorted(cum, CONFIG["dim_variance_target"]) + 1)
    if cum[-1] < CONFIG["dim_variance_target"]:
        m = VAR_MISS.setdefault(layer, {"n": 0, "min": 1.0, "max": 0.0, "kmax": kmax})
        m["n"] += 1
        m["min"], m["max"] = min(m["min"], cum[-1]), max(m["max"], cum[-1])
        if m["n"] == 1:
            log(f"    [{layer}] variance target {CONFIG['dim_variance_target']:.2f} "
                f"NOT reachable within {kmax} components (reached {cum[-1]:.3f}). Capped.")
    k = min(k, kmax)
    rd.components_ = rd.components_[:k]
    rd.explained_variance_ratio_ = rd.explained_variance_ratio_[:k]
    if hasattr(rd, "explained_variance_"):
        rd.explained_variance_ = rd.explained_variance_[:k]
    if hasattr(rd, "singular_values_"):
        rd.singular_values_ = rd.singular_values_[:k]
    rd.n_components = rd.n_components_ = k
    return rd, k


def preprocess_layer(layer, tr, te, source=None, seed=SEED):
    z_flag, topk_flag, freq_flag, knn_flag = RECIPE[layer]
    df = (RAW if source is None else source)[layer]
    A, B = df.iloc[tr], df.iloc[te]

    if knn_flag:
        miss = A.isna().mean()
        keep = miss[miss <= CONFIG["rppa_max_missing"]].index
        A, B = A[keep], B[keep]
        if A.isna().any().any() or B.isna().any().any():
            imp = KNNImputer(n_neighbors=CONFIG["rppa_knn_k"]).fit(A.values)
            A = pd.DataFrame(imp.transform(A.values), index=A.index, columns=A.columns)
            B = pd.DataFrame(imp.transform(B.values), index=B.index, columns=B.columns)
    elif df.isna().any().any():
        med = A.median()
        A, B = A.fillna(med), B.fillna(med)

    if freq_flag:
        fr = A.mean()
        keep = fr[(fr >= CONFIG["mut_freq_lo"]) & (fr <= CONFIG["mut_freq_hi"])].index
        A, B = A[keep], B[keep]

    va = A.var()
    keep = va[va > CONFIG["min_variance"]].index
    A, B = A[keep], B[keep]

    if topk_flag and A.shape[1] > CONFIG["filter_top_k"]:
        sc = score_features(A, CONFIG["filter_stat"])
        A, B = A[sc.nlargest(CONFIG["filter_top_k"]).index], B[sc.nlargest(CONFIG["filter_top_k"]).index]

    n_feat, Atr, Bte = A.shape[1], A.values.astype(np.float64), B.values.astype(np.float64)

    if z_flag:
        sca = StandardScaler().fit(Atr)
        Atr, Bte = sca.transform(Atr), sca.transform(Bte)

    rd, k = reduce_fit(Atr, layer, seed)
    return (rd.transform(Atr).astype(np.float32),
            rd.transform(Bte).astype(np.float32),
            n_feat, float(rd.explained_variance_ratio_.sum()), k)


# ---- CACHE DE RÉDUCTION -- BOUCLE CORRIGÉE ------------------------------
cache = ckpt_load("cache") or {"red": {}, "var": {}, "nfeat": {}, "ncomp": {}}
RED, VARK, NFEAT, NCOMP = cache["red"], cache["var"], cache["nfeat"], cache["ncomp"]
t0 = time.time()
for s, (tr, te) in enumerate(SPLITS):
    if all((s, k) in RED for k in LAYERS):
        continue
    for k in LAYERS:
        Xtr, Xte, nf, vr, nc = preprocess_layer(k, tr, te)
        RED[(s, k)] = (Xtr, Xte)          # <-- CORRIGÉ : tuple, pas 2 cibles séparées
        VARK[(s, k)] = vr
        NFEAT[(s, k)] = nf
        NCOMP[(s, k)] = nc
    if (s + 1) % 5 == 0 or s == len(SPLITS) - 1:
        ckpt_save("cache", {"red": RED, "var": VARK, "nfeat": NFEAT, "ncomp": NCOMP})
        log(f"  cache {s+1}/{len(SPLITS)}  ({time.time()-t0:.0f}s)")

log("\nPer-layer representation (train-fitted, across folds):")
REPR_SUMMARY = {}
for k in LAYERS:
    nf = [NFEAT[(s, k)] for s in range(len(SPLITS))]
    nc = [NCOMP[(s, k)] for s in range(len(SPLITS))]
    vr = [VARK[(s, k)] for s in range(len(SPLITS))]
    REPR_SUMMARY[SHORT[k]] = {"features_median": int(np.median(nf)),
                              "components_median": int(np.median(nc)),
                              "variance_mean": round(float(np.mean(vr)), 4)}
    log(f"{SHORT[k]:<12}feat={int(np.median(nf))}  comp={int(np.median(nc))}  "
        f"var={np.mean(vr)*100:.1f}%")

print("Module 2-3 terminé : cache de réduction construit.")


50 splits = 10 repeats x 5 folds (classifier seed = repeat index)
    [mrna] variance target 0.80 NOT reachable within 120 components (reached 0.792). Capped.
  cache 5/50  (28s)
  cache 10/50  (56s)
  cache 15/50  (83s)
  cache 20/50  (110s)
  cache 25/50  (138s)
  cache 30/50  (167s)
  cache 35/50  (195s)
  cache 40/50  (224s)
  cache 45/50  (252s)
  cache 50/50  (280s)

Per-layer representation (train-fitted, across folds):
Mutations   feat=1599  comp=120  var=68.4%
CNV         feat=5000  comp=16  var=80.4%
mRNA        feat=5000  comp=120  var=79.3%
RPPA        feat=131  comp=35  var=80.3%
Module 2-3 terminé : cache de réduction construit.
